In [2]:
import pandas as pd

## Load and Inspect

In [3]:
us_2y_raw = pd.read_csv("../data/raw/2y_treasury/v1_11-09-25/DGS2.csv")

In [4]:
us_2y_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12856 entries, 0 to 12855
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  12856 non-null  object 
 1   DGS2              12315 non-null  float64
dtypes: float64(1), object(1)
memory usage: 201.0+ KB


In [5]:
us_2y_raw.head()

,observation_date,DGS2
0,1976-06-01,7.26
1,1976-06-02,7.23
2,1976-06-03,7.22
3,1976-06-04,7.12
4,1976-06-07,7.09


## Cleaning

### Column Structuring

In [6]:
us_2y_clean = us_2y_raw.rename(columns={"observation_date": "date", "DGS2": "us_2y"})

In [8]:
us_2y_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12856 entries, 0 to 12855
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    12856 non-null  object 
 1   us_2y   12315 non-null  float64
dtypes: float64(1), object(1)
memory usage: 201.0+ KB


### Data Type Conversions

In [9]:
us_2y_clean.dtypes

date      object
us_2y    float64
dtype: object

In [12]:
us_2y_clean["date"] = pd.to_datetime(us_2y_clean["date"], format="%Y-%m-%d", errors="raise")

In [13]:
us_2y_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12856 entries, 0 to 12855
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    12856 non-null  datetime64[ns]
 1   us_2y   12315 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 201.0 KB


### Handling Null Dates

In [14]:
us_2y_clean[us_2y_clean["date"].isna()]

,date,us_2y


### Indexing

In [15]:
us_2y_clean = us_2y_clean.set_index("date").sort_index()
us_2y_clean.head()

,us_2y
date,
1976-06-01,7.26
1976-06-02,7.23
1976-06-03,7.22
1976-06-04,7.12
1976-06-07,7.09


### Missing Timestamps

In [16]:
full_range = pd.date_range(start=us_2y_clean.index.min(), end=us_2y_clean.index.max(), freq='B')
missing_dates = full_range.difference(us_2y_clean.index)

print("Missing dates:", missing_dates)

Missing dates: DatetimeIndex([], dtype='datetime64[ns]', freq='B')


### Handling Missing Yield Values

Q: How do we want to handle missing values in this time-series? We want to calculate the spread smoothly over time and if either series is missing data then this will cause gaps. Could ff? From a domain perspective, it may be safe enough to assume that if there is no data then the yield for that day is the same as the previous day.

In [18]:
us_2y_clean[us_2y_clean["us_2y"].isna()]

,us_2y
date,
1976-07-05,NaN
1976-09-06,NaN
1976-10-11,NaN
1976-11-02,NaN
1976-11-11,NaN
...,...
2025-04-18,NaN
2025-05-26,NaN
2025-06-19,NaN


In [20]:
us_2y_clean[us_2y_clean["us_2y"].notna()].index.to_series().diff().value_counts()

date
1 days    9593
3 days    2184
4 days     386
2 days     149
5 days       2
Name: count, dtype: int64

Using forward-fill (ffill): Assumes the yield held steady until the next observation. Especially reasonable for short gaps since gilts don’t jump daily outside stress periods.

In [21]:
us_2y_clean["us_2y"] = us_2y_clean["us_2y"].ffill(limit=5)

In [22]:
us_2y_clean[us_2y_clean["us_2y"].isna()]

,us_2y
date,


In [24]:
us_2y_clean[us_2y_clean["us_2y"].isna()]

,us_2y
date,


### Time-Series Validation

In [25]:
us_2y_clean.index[us_2y_clean.index.duplicated()].shape[0] == 0

True

In [26]:
us_2y_clean.index.is_monotonic_increasing

True

## Export

In [27]:
us_2y_clean.to_csv("../data/cleaned/2y_treasury/v1_11-09-25/2y_treasury_cleaned_as_of_110925.csv")
us_2y_clean.dtypes.to_csv("../data/cleaned/2y_treasury/v1_11-09-25/2y_treasury_cleaned_dtypes_as_of_110925.csv")